# RisiKo RL — Training Bot

Notebook di training per il bot RL di RisiKo, da usare su **Colab Pro**.

**Workflow:**
1. Cella 1: monta Google Drive e clona il repo da GitHub
2. Cella 2: installa dipendenze RL
3. Cella 3: verifica che il simulatore funzioni
4. Cella 4: configura il training
5. Cella 5: lancia il training (può durare ore)
6. Cella 6: valuta il bot trainato
7. Cella 7 (opzionale): carica un modello esistente

**IMPORTANTE:** prima di iniziare, sostituisci `IL_TUO_USERNAME` e `il-tuo-repo` nella cella 1 con i tuoi dati GitHub.

## Cella 1: Setup Drive e clona repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_DIR = '/content/drive/MyDrive/risiko-rl-checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoint dir: {CHECKPOINT_DIR}')

# Clona il repo
%cd /content
!rm -rf risiko-rl
!git clone https://github.com/IL_TUO_USERNAME/il-tuo-repo.git risiko-rl
%cd risiko-rl

## Cella 2: Installa dipendenze

In [ ]:
!pip install -r requirements.txt -q
!pip install sb3-contrib -q
!pip install torch -q

import torch
import gymnasium as gym
from sb3_contrib import MaskablePPO
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponibile: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
print(f'Gymnasium: {gym.__version__}')

## Cella 3: Verifica simulatore

In [ ]:
for f in ['test_data', 'test_setup', 'test_motore', 'test_sdadata',
          'test_partita_completa', 'test_encoding', 'test_azioni', 'test_env']:
    print(f'=== {f} ===')
    !python tests/{f}.py 2>&1 | tail -3

## Cella 4: Configura training

In [ ]:
import sys
sys.path.insert(0, '/content/risiko-rl')

from sb3_contrib import MaskablePPO
from sb3_contrib.common.wrappers import ActionMasker
from stable_baselines3.common.vec_env import SubprocVecEnv, VecMonitor
from stable_baselines3.common.callbacks import CheckpointCallback
import numpy as np
import torch

from risiko_env import RisikoEnv

def mask_fn(env):
    return env.get_action_mask()

def make_env(seed):
    def _init():
        env = RisikoEnv(seed=seed)
        env = ActionMasker(env, mask_fn)
        return env
    return _init

TOTAL_TIMESTEPS = 1_000_000
N_ENVS = 8
SEED_BASE = 42

env = SubprocVecEnv([make_env(SEED_BASE + i) for i in range(N_ENVS)])
env = VecMonitor(env)

model = MaskablePPO(
    'MlpPolicy',
    env,
    learning_rate=3e-4,
    n_steps=512,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    verbose=1,
    tensorboard_log=f'{CHECKPOINT_DIR}/tensorboard/',
    device='cuda' if torch.cuda.is_available() else 'cpu',
)
print(f'Modello creato. Device: {model.device}')
print(f'Parametri rete: {sum(p.numel() for p in model.policy.parameters()):,}')

## Cella 5: Lancia training

In [ ]:
checkpoint_callback = CheckpointCallback(
    save_freq=50_000 // N_ENVS,
    save_path=CHECKPOINT_DIR,
    name_prefix='risiko_bot',
)

import time
inizio = time.time()
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=checkpoint_callback,
    progress_bar=True,
)
durata = time.time() - inizio
print(f'Training completato in {durata/60:.1f} minuti')

model.save(f'{CHECKPOINT_DIR}/risiko_bot_finale')
print(f'Modello salvato in {CHECKPOINT_DIR}/risiko_bot_finale.zip')

## Cella 6: Valuta il bot trainato

In [ ]:
from risiko_env import RisikoEnv
import numpy as np

def valuta_bot(model, n_partite=100, verbose=False):
    vittorie = 0
    rewards_totali = []
    durate_step = []

    for seed in range(n_partite):
        env = RisikoEnv(seed=seed)
        obs, info = env.reset()
        terminated = False
        truncated = False
        step_count = 0

        while not (terminated or truncated):
            mask = info['action_mask']
            action, _ = model.predict(obs, action_masks=mask, deterministic=True)
            obs, reward, terminated, truncated, info = env.step(int(action))
            step_count += 1

        rewards_totali.append(reward)
        durate_step.append(step_count)
        if reward == 1.0:
            vittorie += 1

        if verbose and seed < 5:
            print(f'Partita {seed}: reward={reward}, step={step_count}, '
                  f'vincitore={info["vincitore"]}')

    win_rate = vittorie / n_partite
    reward_medio = np.mean(rewards_totali)
    step_medio = np.mean(durate_step)

    return {
        'win_rate': win_rate,
        'reward_medio': reward_medio,
        'step_medio': step_medio,
    }

risultati = valuta_bot(model, n_partite=100, verbose=True)
print()
print(f'Win rate: {risultati["win_rate"]*100:.1f}% (atteso ~25% per bot random)')
print(f'Reward medio: {risultati["reward_medio"]:.3f}')
print(f'Step medio: {risultati["step_medio"]:.0f}')

## Cella 7 (opzionale): Carica modello esistente

In [ ]:
import glob
checkpoints = sorted(glob.glob(f'{CHECKPOINT_DIR}/risiko_bot_*.zip'))
if checkpoints:
    print(f'Checkpoints disponibili:')
    for c in checkpoints:
        print(f'  {c}')
    model_loaded = MaskablePPO.load(checkpoints[-1], env=env)
    print(f'\nCaricato: {checkpoints[-1]}')
else:
    print('Nessun checkpoint trovato')